# Mid Fusion Lite - Quick Ablation trên Kaggle
Notebook này giúp bạn chạy nhanh kịch bản so sánh cấu hình trực tiếp trên Kaggle.

## Hướng dẫn:
1. Thêm dataset chứa code của bạn (ví dụ: `khoangdz/isic-2024`)
2. Thêm dataset cuộc thi (`isic-2024-challenge`)
3. Bật Internet trong Kaggle (để cài albumentations, timm, ...)
4. Nhấn **Run All**!

In [ ]:
# =========================
# CELL 1: SETUP KAGGLE PATHS
# =========================
from pathlib import Path
import sys

# ===== KAGGLE PATHS =====
KAGGLE_CODE_DATASET_ROOT = '/kaggle/input/datasets/khoangdz/isic-2024'
KAGGLE_COMP_DATASET_ROOT = '/kaggle/input/competitions/isic-2024-challenge'
KAGGLE_OUTPUT_DIR = '/kaggle/working/ablation_outputs'

is_kaggle = Path('/kaggle/input').exists()

if is_kaggle:
    CODE_DATASET_ROOT = KAGGLE_CODE_DATASET_ROOT
else:
    CODE_DATASET_ROOT = '.' # Fallback de chay thu tren local neu can

# Them thu muc code vao sys.path de import duoc
if CODE_DATASET_ROOT not in sys.path:
    sys.path.append(CODE_DATASET_ROOT)

print('✅ Môi trường:', 'KAGGLE' if is_kaggle else 'LOCAL')
print('✅ Đường dẫn code:', CODE_DATASET_ROOT)

In [ ]:
# =========================
# CELL 2: INSTALL DEPENDENCIES
# =========================
!pip -q install -r "{CODE_DATASET_ROOT}/requirements.txt"
!pip -q install albumentations timm torchmetrics h5py

In [ ]:
# =========================
# CELL 3: ĐỊNH NGHĨA EXPERIMENTS & CHẠY
# =========================
from quick_ablation import run_phase1, run_phase2

# Bạn có thể tuỳ ý thêm/sửa cấu hình ngay tại đây mà không cần upload lại code!
MY_EXPERIMENTS = {
    # --- 1. SO SÁNH SỨC MẠNH TỪNG NHÁNH ---
    'baseline': {},
    'mod_tabular_only': {'modality_mode': 'tabular_only'},
    'mod_image_only': {'modality_mode': 'image_only'},
    
    # --- 2. TỐI ƯU NHÁNH BẢNG (Chạy siêu nhanh) ---
    # (Bỏ comment để chạy)
    #'tab_k8': {'modality_mode': 'tabular_only', 'tabm_k': 8},
    #'tab_k32': {'modality_mode': 'tabular_only', 'tabm_k': 32},
    #'tab_hidden_256': {'modality_mode': 'tabular_only', 'tab_hidden_dim': 256},
    #'tab_blocks_2': {'modality_mode': 'tabular_only', 'tabm_n_blocks': 2},
    
    # --- 3. TỐI ƯU LEARNING RATE & DROPOUT CHO BẢNG ---
    #'tab_lr_1e4': {'modality_mode': 'tabular_only', 'lr': 1e-4},
    #'tab_lr_5e4': {'modality_mode': 'tabular_only', 'lr': 5e-4},
    #'tab_wd_5e4': {'modality_mode': 'tabular_only', 'weight_decay': 5e-4},
    #'tab_dropout_0.2': {'modality_mode': 'tabular_only', 'tab_dropout': 0.2},
    
    # --- 4. TỐI ƯU MẤT CÂN BẰNG DỮ LIỆU ---
    #'neg_ratio_4': {'modality_mode': 'tabular_only', 'negative_ratio': 4},
    #'neg_ratio_8': {'modality_mode': 'tabular_only', 'negative_ratio': 8},
    
    # --- 5. TỐI ƯU NHÁNH ẢNH ---
    # (Nhánh ảnh sẽ chạy chậm hơn)
    #'img_b1': {'modality_mode': 'image_only', 'backbone_name': 'efficientnet_b1'},
    #'img_size_160': {'modality_mode': 'image_only', 'image_size': 160},
}

print("BẮT ĐẦU PHASE 1 (Sàng lọc nhanh 1 fold)...")
phase1_results = run_phase1(MY_EXPERIMENTS)

# Lấy ra top 3 để chạy Phase 2 (3 fold xác nhận)
import math
valid_results = [(n, p, t) for n, p, t in phase1_results if not math.isnan(p)]
sorted_results = sorted(valid_results, key=lambda x: -x[1])
top_3_names = [name for name, _, _ in sorted_results[:3]]

print(f"\nBẮT ĐẦU PHASE 2 cho Top 3: {top_3_names}...")
phase2_results = run_phase2(MY_EXPERIMENTS, top_3_names)
